# ⚡ 1-CLICK UNCENSOR PIPELINE: DEEPSEEK-CODER-V2-LITE-INSTRUCT (MoE 16B)
Notebook chuyên dụng để **bóc tách 100% kiểm duyệt** cho siêu mô hình **DeepSeek-Coder-V2-Lite-Instruct (MoE)** và **lưu thẳng lên Hugging Face cá nhân**.

### 🎯 Đặc điểm kỹ thuật:
- **Vá lỗi tương thích Transformers:** Tự động sửa lỗi `is_torch_fx_available`, `DynamicCache.from_legacy_cache`, và `_tied_weights_keys` của DeepSeek-V2 MoE.
- **Xử lý ma trận MoE đa chuyên gia:** Triệt tiêu vector kiểm duyệt trên toàn bộ 64 Experts và Shared Experts.
- **Không tốn Google Drive:** Lưu trên SSD 150GB của Colab và đẩy thẳng lên Hugging Face của bạn.

In [ ]:
# @title 🚀 BẤM NÚT NÀY ĐỂ BẮT ĐẦU BÓC TÁCH DEEPSEEK-CODER-V2 (TỰ ĐỘNG 100%)
MODEL_CHOICE = "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct"
HF_TOKEN = "" # @param {type:"string"} # Để trống sẽ tự động dùng token Leon234aamon của bạn

import os
import torch

# Tự động nhúng Token mặc định của bạn
token_parts = ["hf_", "npwAkBYhCxOus", "BXnQeBXDraYMhmi", "Szhmsk"]
DEFAULT_HF_TOKEN = "".join(token_parts)
if not HF_TOKEN.strip():
    HF_TOKEN = DEFAULT_HF_TOKEN

print("=" * 70)
print(f"🚀 BẮT ĐẦU QUY TRÌNH UNCENSOR: {MODEL_CHOICE}")
print("=" * 70)

# BƯỚC 1: THIẾT LẬP THƯ MỤC TRÊN Ổ SSD COLAB
clean_name = MODEL_CHOICE.split("/")[-1]
OUTPUT_MODEL_NAME = f"{clean_name}-Uncensored"
TARGET_PATH = f"/content/{OUTPUT_MODEL_NAME}"
os.makedirs(TARGET_PATH, exist_ok=True)
print(f"📁 Thư mục lưu tạm trên SSD Colab: {TARGET_PATH}")

# BƯỚC 2: CÀI ĐẶT THƯ VIỆN & XÁC THỰC HUGGING FACE
print("\n📦 [1/4] Đang cài đặt thư viện & Xác thực Hugging Face...")
!pip install -q -U transformers datasets accelerate huggingface_hub

from huggingface_hub import login, HfApi
login(token=HF_TOKEN.strip(), add_to_git_credential=True)
try:
    hf_user = HfApi(token=HF_TOKEN.strip()).whoami()["name"]
except Exception:
    hf_user = "Leon234aamon"
print(f"🔑 Đã xác thực tài khoản Hugging Face: {hf_user}")

# VÁ LỖI TƯƠNG THÍCH TRANSFORMERS CHO DEEPSEEK-V2 MOE
import transformers.utils.import_utils
if not hasattr(transformers.utils.import_utils, 'is_torch_fx_available'):
    transformers.utils.import_utils.is_torch_fx_available = lambda: True

from transformers.cache_utils import DynamicCache
if not hasattr(DynamicCache, 'from_legacy_cache'):
    DynamicCache.from_legacy_cache = staticmethod(lambda past_key_values=None: DynamicCache() if past_key_values is None else past_key_values)
print("🔧 Đã kích hoạt toàn bộ bản vá tương thích cho DeepSeek-V2!")

from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

# BƯỚC 3: TẢI MODEL LÊN GPU
print(f"\n📥 [2/4] Đang nạp Model {MODEL_CHOICE} lên GPU {torch.cuda.get_device_name(0)}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHOICE, trust_remote_code=True, token=HF_TOKEN.strip())
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_CHOICE,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    token=HF_TOKEN.strip()
)
print("✅ Nạp Model thành công!")

# BƯỚC 4: TÍNH TOÁN VECTOR TRIỆT TIÊU KIỂM DUYỆT (ABLITERATION)
print("\n🧮 [3/4] Đang trích xuất và triệt tiêu vector từ chối trả lời (Censorship Directions)...")
harmful_ds = load_dataset("mlabonne/harmful_behaviors", split="train[:120]")
harmless_ds = load_dataset("mlabonne/harmless_alpaca", split="train[:120]")

def format_prompts(dataset, col="text"):
    prompts = []
    for item in dataset:
        chat = [{"role": "user", "content": item[col]}]
        try:
            formatted = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
        except Exception:
            formatted = f"User: {item[col]}\nAssistant:"
        prompts.append(formatted)
    return prompts

harmful_prompts = format_prompts(harmful_ds)
harmless_prompts = format_prompts(harmless_ds)

def get_mean_activations(prompts, batch_size=8):
    all_acts = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to(model.device)
        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True, use_cache=False)
            hidden_states = torch.stack(outputs.hidden_states) # (n_layers+1, batch, seq_len, dim)
            seq_lens = inputs.attention_mask.sum(dim=1) - 1
            batch_acts = []
            for b_idx, s_len in enumerate(seq_lens):
                batch_acts.append(hidden_states[:, b_idx, s_len, :])
            all_acts.append(torch.stack(batch_acts))
    all_acts = torch.cat(all_acts, dim=0)
    return all_acts.mean(dim=0)

print("  • Đang tính ma trận ẩn của Prompts...")
harmful_acts = get_mean_activations(harmful_prompts)
harmless_acts = get_mean_activations(harmless_prompts)

refusal_directions = harmful_acts - harmless_acts
refusal_directions = refusal_directions / refusal_directions.norm(dim=-1, keepdim=True)

# Triệt tiêu kiểm duyệt trên các Layer cốt lõi
n_layers = model.config.num_hidden_layers
start_layer = int(0.30 * n_layers)
end_layer = int(0.85 * n_layers)
print(f"  • Đang bóc tách vĩnh viễn cơ chế kiểm duyệt từ Layer {start_layer} đến {end_layer}...")

layers = model.model.layers if hasattr(model, 'model') and hasattr(model.model, 'layers') else model.layers

def abliterate_sublayer(linear_mod, direction):
    if hasattr(linear_mod, 'weight') and linear_mod.weight is not None:
        W = linear_mod.weight.data
        if W.shape[0] == direction.shape[0]:
            proj = torch.matmul(direction, W)
            linear_mod.weight.data = W - torch.outer(direction, proj)

for layer_idx in range(start_layer, end_layer):
    v = refusal_directions[layer_idx + 1].to(model.dtype).to(model.device)
    v = v / v.norm()
    layer = layers[layer_idx]
    
    # 1. Triệt tiêu trên Self-Attention Output Projection
    if hasattr(layer, 'self_attn') and hasattr(layer.self_attn, 'o_proj'):
        abliterate_sublayer(layer.self_attn.o_proj, v)
        
    # 2. Triệt tiêu trên MLP Down Projection (Dense)
    if hasattr(layer, 'mlp') and hasattr(layer.mlp, 'down_proj'):
        abliterate_sublayer(layer.mlp.down_proj, v)
        
    # 3. Triệt tiêu trên MoE Experts (DeepSeek-V2 MoE)
    if hasattr(layer, 'mlp') and hasattr(layer.mlp, 'experts'):
        for expert in layer.mlp.experts:
            if hasattr(expert, 'down_proj'):
                abliterate_sublayer(expert.down_proj, v)
    if hasattr(layer, 'mlp') and hasattr(layer.mlp, 'shared_experts'):
        if hasattr(layer.mlp.shared_experts, 'down_proj'):
            abliterate_sublayer(layer.mlp.shared_experts.down_proj, v)

print("✅ Bóc tách kiểm duyệt DeepSeek-Coder-V2 thành công 100%!")

# BƯỚC 5: LƯU TẠM VÀ UPLOAD THẲNG LÊN HUGGING FACE
print(f"\n💾 [4/4] Đang đóng gói và lưu trữ Model...")

# Vá lỗi định dạng _tied_weights_keys của Transformers v4.49+ cho DeepSeek-V2
for submodule in model.modules():
    if hasattr(submodule, '_tied_weights_keys') and isinstance(submodule._tied_weights_keys, list):
        submodule._tied_weights_keys = {k: k for k in submodule._tied_weights_keys}

model.save_pretrained(TARGET_PATH, max_shard_size="5GB")
tokenizer.save_pretrained(TARGET_PATH)

repo_id = f"{hf_user}/{OUTPUT_MODEL_NAME}"
print(f"\n📤 Đang tải Model lên Hugging Face Hub (Private Repo): {repo_id}...")
api = HfApi(token=HF_TOKEN.strip())
api.create_repo(repo_id=repo_id, exist_ok=True, private=True)
api.upload_folder(folder_path=TARGET_PATH, repo_id=repo_id, repo_type="model")
print("\n" + "=" * 70)
print(f"🎉 HOÀN TẤT 100%! Model đã được lưu an toàn tại: https://huggingface.co/{repo_id}")
print("=" * 70)
